In [1]:
from functools import lru_cache

sentences = [
    "Cảm giác bước ra khỏi phòng điều hòa lúc 12 giờ trưa quả thực là một cú sốc vật lý. Cái nắng tháng sáu dội xuống thành phố Hà Nội như một chảo lửa khổng lồ, hầm hập và ngột ngạt đến nghẹt thở. Mồ hôi túa ra ròng ròng chỉ sau năm phút đứng chờ đèn đỏ, khiến người ta trôi tuột mọi sự lãng mạn và chỉ muốn thốt lên: 'ĐM mùa hè!''.",
    "Kẹt xe ở ngã tư Sở giữa giờ tan tầm luôn là một trải nghiệm kinh hoàng, nhưng nó còn tệ hơn gấp bội vào những ngày nhiệt độ chạm ngưỡng 40 độ. Hơi nóng từ mặt đường nhựa và bô xe máy phả thẳng vào mặt, nung chín sự kiên nhẫn của hàng vạn con người. Giữa thành phố Hà Nội chật chội này, tiếng còi xe inh ỏi dường như cũng đang gào thét 'ĐM mùa hè' thay cho những khuôn mặt nhăn nhó, mệt mỏi.",
    "Những quán trà đá vỉa hè quen thuộc của thành phố Hà Nội hôm nay cũng chẳng thể xoa dịu nổi cái oi ả. Cốc nước nhân trần đá vừa mang ra đã vã mồ hôi hột, đá tan nhanh hơn cả tốc độ uống. Anh bạn ngồi cạnh tôi cởi vội hai cúc áo sơ mi, quệt những giọt mồ hôi đang lăn dài trên trán và làu bàu: 'ĐM mùa hè, nóng điên lên được!'.",
    "Ngay cả khi màn đêm buông xuống, sự oi bức vẫn ngoan cố bám trụ lấy thành phố Hà Nội. Những khối bê tông san sát nhau sau một ngày phơi mình dưới nắng gắt giờ bắt đầu tỏa nhiệt, biến không khí trở thành một phòng xông hơi khổng lồ. Mở cửa sổ đón gió mà chỉ thấy hơi nóng ập vào mặt, đành chậc lưỡi đóng sập lại và lầm bầm 'ĐM mùa hè' trước khi với tay bật điều hòa.",
    "Từng cơn gió thổi qua mặt nước Hồ Tây tưởng như sẽ mang theo hơi nước dịu mát, nhưng thực tế lại hắt cái hơi nóng hầm hập táp thẳng vào da thịt. Bầu trời thành phố Hà Nội những ngày này trong vắt không một gợn mây, lên ảnh thì đẹp thật đấy, nhưng cái giá phải trả cho việc phơi nắng ngoài đường thì đúng là 'ĐM mùa hè'.",
    "Bất chấp sự khắc nghiệt của thời tiết, guồng quay cuộc sống vẫn không thể dừng lại. Những cô chú bán hàng rong, những tài xế xe ôm công nghệ vẫn đang gồng mình chống chọi với cái nắng rát da rát thịt của thành phố Hà Nội. Dù mệt nhoài và thỉnh thoảng lại cáu bẳn buông một tiếng chửi thề 'ĐM mùa hè', họ vẫn bám mặt đường mưu sinh, chỉ mong ngày dài mau tắt nắng để được về nhà ngả lưng.",

]

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def _ensure_sentence_transformers():
    try:
        from sentence_transformers import SentenceTransformer
    except ModuleNotFoundError:
        import subprocess
        import sys

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "sentence-transformers",
        ])
        from sentence_transformers import SentenceTransformer

    return SentenceTransformer


@lru_cache(maxsize=1)
def _load_model(model_name=MODEL_NAME):
    SentenceTransformer = _ensure_sentence_transformers()
    return SentenceTransformer(model_name, device="cpu")


def embedding(texts, model_name=MODEL_NAME, normalize=True):
    """Embed one sentence or a list of sentences on CPU."""
    is_single_text = isinstance(texts, str)
    if is_single_text:
        texts = [texts]

    model = _load_model(model_name)
    vectors = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
        show_progress_bar=False,
        device="cpu",
    )
    return vectors[0] if is_single_text else vectors




def cosine(sentence_a, sentence_b, model_name=MODEL_NAME):
    """Return cosine similarity between two sentences."""
    import numpy as np

    vec_a, vec_b = embedding(
        [sentence_a, sentence_b],
        model_name=model_name,
        normalize=False,
    )
    denominator = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denominator == 0:
        return 0.0

    return float(np.dot(vec_a, vec_b) / denominator)


def cosine_vectors(vec_a, vec_b):
    """Return cosine similarity between two embedding vectors."""
    import numpy as np

    denominator = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denominator == 0:
        return 0.0

    return float(np.dot(vec_a, vec_b) / denominator)
print("number of sentences:", len(sentences))
embeddings = embedding(sentences)
print("embedding shape:", embeddings.shape)
print("first 8 dims:\n", embeddings[:, :8])

similarity = cosine(sentences[0], sentences[1])
print("cosine similarity sentences[0] vs sentences[1]:", similarity)


Loading weights: 100%|##########| 199/199 [00:00<00:00, 7480.36it/s]


number of sentences: 6
embedding shape: (6, 384)
first 8 dims:
 [[ 0.06965218  0.09590678  0.0343086   0.10057794  0.09530097 -0.03524459
   0.04972631  0.02152576]
 [ 0.08982197  0.06864246  0.01894003  0.08416487  0.09413788 -0.02709338
   0.05026587  0.03984588]
 [ 0.04834234  0.09074012  0.04495667  0.084089    0.07270593 -0.03439096
   0.09975301  0.00782761]
 [ 0.05209164  0.12336107  0.04452225  0.10852534  0.09050093 -0.02890241
  -0.00415511  0.06296588]
 [ 0.07169154  0.07114076  0.0403127   0.05550445  0.13374443 -0.04446241
   0.02675941  0.01268227]
 [ 0.094019    0.07703307  0.02454677  0.09183396  0.11818168 -0.00541787
   0.02610342  0.02782381]]
cosine similarity sentences[0] vs sentences[1]: 0.6969041228294373


In [2]:
query = "thủ đô của Việt Nam là ở đâu?"

em_q = embedding(query)
print("query embedding shape:", em_q.shape)


query embedding shape: (384,)


In [3]:
scores = [cosine_vectors(em_q, chunk_embedding) for chunk_embedding in embeddings]
most_sim_idx = max(range(len(scores)), key=scores.__getitem__)
most_sim_score = scores[most_sim_idx]
most_sim_sentence = sentences[most_sim_idx]

print("most similar index:", most_sim_idx)
print("most similar score:", most_sim_score)
print("most similar sentence:", most_sim_sentence)


most similar index: 3
most similar score: 0.4354759454727173
most similar sentence: Ngay cả khi màn đêm buông xuống, sự oi bức vẫn ngoan cố bám trụ lấy thành phố Hà Nội. Những khối bê tông san sát nhau sau một ngày phơi mình dưới nắng gắt giờ bắt đầu tỏa nhiệt, biến không khí trở thành một phòng xông hơi khổng lồ. Mở cửa sổ đón gió mà chỉ thấy hơi nóng ập vào mặt, đành chậc lưỡi đóng sập lại và lầm bầm 'ĐM mùa hè' trước khi với tay bật điều hòa.


In [4]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_API_KEY_ENV = "OPENROUTER_API_KEY"


def _load_env_file(env_path=".env"):
    """Load KEY=VALUE pairs from .env into os.environ without printing secrets."""
    import os
    from pathlib import Path

    path = Path(env_path)
    if not path.exists():
        return

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("\'").strip('"')
        if key and key not in os.environ:
            os.environ[key] = value


def _get_openrouter_api_key(required=True):
    import os

    _load_env_file()
    api_key = os.getenv(OPENROUTER_API_KEY_ENV)
    if required and not api_key:
        raise ValueError(f"Set {OPENROUTER_API_KEY_ENV} in your environment or .env before calling answer().")

    return api_key


def _ensure_openai_sdk():
    try:
        from openai import OpenAI
    except ModuleNotFoundError:
        import subprocess
        import sys

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "openai",
        ])
        from openai import OpenAI

    return OpenAI


def answer(query, most_sim, model=OPENROUTER_MODEL, max_tokens=512):
    """Answer query using OpenRouter and the most similar retrieved sentence as context."""
    OpenAI = _ensure_openai_sdk()
    client = OpenAI(
        base_url=OPENROUTER_BASE_URL,
        api_key=_get_openrouter_api_key(),
    )
    completion = client.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "system",
                "content": (
                    "Bạn là trợ lý trả lời câu hỏi bằng tiếng Việt. "
                    "Chỉ dùng CONTEXT được cung cấp. "
                    "Nếu CONTEXT không đủ thông tin để trả lời, hãy nói rõ là không tìm thấy thông tin trong context."
                ),
            },
            {
                "role": "user",
                "content": f"""QUESTION:
{query}

CONTEXT:
{most_sim}

ANSWER:""",
            },
        ],
    )
    return completion.choices[0].message.content.strip()


if "most_sim_sentence" in globals():
    print("ready to answer with context index:", most_sim_idx)
    if _get_openrouter_api_key(required=False):
        final_answer = answer(query, most_sim_sentence)
        print("answer:", final_answer)
    else:
        print(f"{OPENROUTER_API_KEY_ENV} is not set in env or .env, so answer() was defined but not called.")


ready to answer with context index: 3
answer: Thủ đô của Việt Nam là Hà Nội.
